# Lab 1E: Data Modeling / Partition Keys in Python

**Time**: ~60 min  
**Environment**: Jupyter kernel in VS Code  

In this exercise you will explore partition key strategies, composite partition keys, denormalized fan-out patterns, and TTL policies in Azure Cosmos DB.

The lab follows the same structure as the C# version. Run each cell in order to complete the steps.

In [ ]:
%pip install azure-cosmos azure-identity python-dotenv --quiet

## Step 0: Initialize Connection

Set up the Cosmos client connection to the `Modeling` database on the **provisioned-throughput** Cosmos account. Lab 1E uses provisioned throughput so per-partition RU metrics are available in Azure Monitor for the Step 6 comparison. The database is named separately from the serverless account's `WorkshopData` to avoid confusion when both endpoints are configured.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os

ENDPOINT = os.environ.get("COSMOS_ENDPOINT_PROVISIONED")
DB_NAME = "Modeling"

if not ENDPOINT:
    raise RuntimeError("COSMOS_ENDPOINT_PROVISIONED environment variable is required (see SetEnv.ps1).")

print(f"Cosmos Endpoint: {ENDPOINT}")
print(f"Database: {DB_NAME}")

In [ ]:
from azure.cosmos import CosmosClient, PartitionKey
from azure.identity import DefaultAzureCredential

cred = DefaultAzureCredential()
client = CosmosClient(url=ENDPOINT, credential=cred)
db = client.get_database_client(DB_NAME)
print(f"Connected to: {ENDPOINT}/{DB_NAME}")

In [ ]:
import asyncio
from azure.cosmos.aio import CosmosClient as AsyncCosmosClient
from azure.cosmos.exceptions import CosmosHttpResponseError

# Shared seeding config — matches the C# lab so both versions drive the same traffic.
ORDER_COUNT = 10000
CONCURRENCY = 64
STATUSES = ["pending", "shipped", "delivered"]
# ~1 KB filler so each write costs ~10 RU instead of ~5 RU; needed to make the
# hot-partition pattern unmistakable on the 'Normalized RU Consumption' chart.
FILLER = "x" * 1024

def build_order(i, customer_id, order_date, partition_key=None):
    order = {
        "id": f"order_{i}",
        "customerId": customer_id,
        "orderDate": order_date,
        "total": round(10 + (i * 3.33), 2),
        "status": STATUSES[i % 3],
        "items": [
            {"sku": f"SKU_{i%5}", "qty": (i % 3) + 1},
            {"sku": f"SKU_{(i+1)%5}", "qty": ((i + 1) % 3) + 1},
        ],
        "notes": FILLER,
    }
    if partition_key is not None:
        order["partitionKey"] = partition_key
    return order

async def seed_orders(container_name, orders):
    sem = asyncio.Semaphore(CONCURRENCY)
    completed = 0
    start = asyncio.get_event_loop().time()

    async with AsyncCosmosClient(url=ENDPOINT, credential=cred) as async_client:
        container = async_client.get_database_client(DB_NAME).get_container_client(container_name)

        async def upsert(order):
            nonlocal completed
            async with sem:
                try:
                    await container.upsert_item(body=order)
                except CosmosHttpResponseError as ex:
                    if ex.status_code == 429:
                        await asyncio.sleep(float(ex.headers.get("x-ms-retry-after-ms", 1000)) / 1000)
                        await container.upsert_item(body=order)
                    else:
                        raise
                completed += 1
                if completed % 1000 == 0:
                    elapsed = asyncio.get_event_loop().time() - start
                    print(f"  ...{completed}/{len(orders)} ({elapsed:.1f}s elapsed)")

        await asyncio.gather(*(upsert(o) for o in orders))

    elapsed = asyncio.get_event_loop().time() - start
    print(f"Wrote {len(orders)} orders to '{container_name}' in {elapsed:.1f}s")

## Step 1: Seed Data with Hot Partition (Prebuilt)

Seed 100 orders into the pre-deployed `OrdersHot` container — all with the same `customerId` to simulate a hot partition scenario.

> **Note**: Containers are deployed in advance via the workshop Bicep template (`CosmosLabs2026/bicep/modules/cosmosdb.bicep`) because Cosmos DB AAD tokens only authorize data-plane operations, not container CRUD.

In [ ]:
hot_container_name = "OrdersHot"
hot_container = db.get_container_client(hot_container_name)

# Every order uses customerId 'CUST_001', so all writes hit one logical partition.
orders = [
    build_order(
        i,
        customer_id="CUST_001",
        order_date=f"2026-01-{(i % 28)+1:02d}",
    )
    for i in range(ORDER_COUNT)
]

print(f"Seeding {len(orders)} orders into '{hot_container_name}' (all customerId='CUST_001')")
print(f"Using {CONCURRENCY} concurrent writers to drive sustained RU on the hot partition...")

await seed_orders(hot_container_name, orders)

print("Note: All orders use the same partition key 'CUST_001' - this creates a hot partition.")
print("Check Azure Portal > Cosmos DB > Metrics > 'Normalized RU Consumption (Max)' split by PartitionKeyRangeId")

## Step 2: Inspect Composite-Key Container (STUDENT EXERCISE)

Read the pre-deployed `OrdersComposite` container and confirm it's keyed on `/partitionKey`. The lab uses a **synthetic composite key**: each document writes `customerId#orderDate` (e.g. `CUST_001#2026-01-15`) into `/partitionKey` to spread load across partitions.

**Expected output**: container is keyed on `/partitionKey` and verification message prints.

**Hint**: `db.get_container_client(name).read()` returns a dict including `partitionKey.paths`.

In [ ]:
composite_container_name = "OrdersComposite"

orders_composite = db.get_container_client(composite_container_name)
props = orders_composite.read()
pk_paths = props["partitionKey"]["paths"]

print(f"Container '{composite_container_name}' found")
print(f"  Partition key paths: {pk_paths}")

if "/partitionKey" in pk_paths:
    print("  Composite container verified (synthetic '/partitionKey').")
else:
    print("  WARNING: expected partition key path '/partitionKey' not present.")

## Step 3: Re-seed with Composite Partition Key (STUDENT EXERCISE)

Update each order to include a composite partition key value, then re-seed into the new container.

**Expected output**: All 100 orders re-seeded into 'orders_composite'.

**Hint**: The composite key value format is `customerId#orderDate` (e.g., `CUST_001#2026-01-15`). Use `PartitionKey(value)` to pass it.

In [ ]:
from datetime import date, timedelta

# Synthetic composite key 'customerId#orderDate' spreads the same 10k writes
# across 50 customers x 100 dates = up to 5000 logical partitions.
base_date = date(2026, 1, 1)
composite_orders = []
for i in range(ORDER_COUNT):
    customer_id = f"CUST_{(i % 50):03d}"
    order_date = (base_date + timedelta(days=i % 100)).isoformat()
    composite_orders.append(build_order(
        i,
        customer_id=customer_id,
        order_date=order_date,
        partition_key=f"{customer_id}#{order_date}",
    ))

print(f"Re-seeding {len(composite_orders)} orders into '{composite_container_name}' (composite '/partitionKey')")
print(f"Using {CONCURRENCY} concurrent writers...")

await seed_orders(composite_container_name, composite_orders)

print("Note: Same write volume as Step 1, but spread across ~5000 logical partitions.")
print("Difference in run time between the two containers is often visible immediately.")

## Step 4: Denormalized Fan-Out Pattern (STUDENT EXERCISE)

Query for denormalized data — retrieve order items stored inline within an order document.

**Expected output**: The `items` array from `order_0`.

**Hint**: Use `SELECT VALUE c.items FROM c WHERE c.id = 'order_0'` to retrieve the inline items.

In [ ]:
fan_out_query = "SELECT VALUE c.items FROM c WHERE c.id = 'order_0'"

results = list(hot_container.query_items(
    query=fan_out_query,
    enable_cross_partition_query=True
))

fan_out_ru = float(hot_container.client_connection.last_response_headers["x-ms-request-charge"])

print("Fan-out items from order_0:")
for item in results:
    print(f"  Items: {item}")
print(f"RU charged: {fan_out_ru}")

## Step 5: Inspect TTL Container (STUDENT EXERCISE)

Read the pre-deployed `EventsTtl` container and confirm `defaultTtl` is set to 2,592,000 seconds (30 days). Documents in this container are automatically deleted 30 days after their last modification.

**Expected output**: `TTL verified (2592000s = 30 days).`

**Hint**: `props["defaultTtl"]` on the container properties dict.

In [ ]:
ttl_container_name = "EventsTtl"
thirty_days_in_seconds = 30 * 24 * 60 * 60  # 2,592,000 seconds

ttl_container = db.get_container_client(ttl_container_name)
props = ttl_container.read()
ttl = props.get("defaultTtl")

print(f"Container '{ttl_container_name}' found")
print(f"  defaultTtl: {ttl} seconds")

if ttl == thirty_days_in_seconds:
    print(f"  TTL verified ({thirty_days_in_seconds}s = 30 days).")
else:
    print(f"  WARNING: expected defaultTtl = {thirty_days_in_seconds} (30 days).")

## Step 6: Compare RU Distribution

The hot partition container concentrates all write RU on a single partition. The composite partition container distributes writes across multiple partitions.

Check **Azure Portal** > Cosmos DB > Monitor > Metrics > Request Units > Partition-Key to see:
- **Hot partition container**: RU spike on `CUST_001`
- **Composite container**: flat distribution across partitions

In [ ]:
print("Hot partition container should show spike in Azure Monitor")
print("Composite container should show flat distribution across partitions")
print()
print("Check Azure Portal > Cosmos DB > Monitor > Metrics > Request Units > Partition-Key")

print("\n=== Lab Complete ===")
print("You have completed the Data Modeling exercise in Python. You:")
print("- Seeded a container with a single partition key value (hot partition scenario)")
print("- Inspected a container keyed on a synthetic composite partition key")
print("- Re-seeded data using composite partition key values")
print("- Queried denormalized fan-out data using SELECT VALUE")
print("- Inspected a TTL policy that automates document expiration")
print("\nKey takeaways:")
print("- Composite partition keys distribute load across partitions")
print("- Denormalization (fan-out) reduces cross-partition queries")
print("- TTL policies automate data lifecycle management")